# Module 1 · Real-Time Event Streaming & Live Semantic Layers

**Phase:** Data & Systems Foundations  
**Prerequisites:** NB08 (Spark pipelines), NB09 (Data Orchestration)
**Toolchain:** Apache Kafka · Apache Flink · Debezium CDC  
**Objective:** Build event-driven pipelines that continuously feed AI systems with millisecond-fresh features and live vector embeddings.

---

## Why Batch Processing Is Not Enough for AI

**Scenario:** A fraud detection model runs hourly batch inference. A fraudster makes 50 transactions in 30 minutes. The model doesn't see them until the NEXT batch run — by then, $50,000 is gone.

**With streaming:** Every transaction hits the model in <100ms. Transaction #3 is flagged and blocked.

| | Batch | Micro-Batch (Spark) | True Streaming (Flink) |
|---|---|---|---|
| Latency | Minutes-hours | 1-10 seconds | 1-50 milliseconds |
| Best for | Reports, training data | Near-real-time features | Fraud, trading, live personalization |
| Complexity | Low | Medium | High |
| Cost | Low | Medium | Higher |

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Batch processing is too slow for modern AI. Seniors build event-driven architectures where data flows continuously like water.

**Mental Model:** Kafka is a massive, indestructible conveyor belt of events. Flink is the robotic arm reading items off the belt, transforming them instantly, and placing them on a new belt.

**Common Junior Pitfall:** Trying to build a "streaming" system using cron jobs that run every 1 minute. Real streaming requires persistent open connections and state management.

---


## 📑 Table of Contents

* [Why Batch Processing Is Not Enough for AI](#why-batch-processing-is-not-enough-for-ai)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Apache Kafka: The Event Backbone](#1--apache-kafka-the-event-backbone)
  * [Key Concepts](#key-concepts)
  * [1.1 Kafka Producer & Consumer Implementation](#11-kafka-producer--consumer-implementation)
* [2 · Apache Flink: Stateful Stream Processing](#2--apache-flink-stateful-stream-processing)
  * [Flink SQL & Windowing Concepts](#flink-sql--windowing-concepts)
* [3 · CDC with Debezium: Streaming Database Changes](#3--cdc-with-debezium-streaming-database-changes)
  * [3.1 Handling CDC Payloads](#31-handling-cdc-payloads)
* [4 · Live Semantic Layer: Real-Time Vector Embeddings](#4--live-semantic-layer-real-time-vector-embeddings)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


In [ ]:
# Cell 1 — Install dependencies
!pip install -q confluent-kafka pandas numpy
import time, json, random
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
print("✅ Streaming environment ready!")


---
## 1 · Apache Kafka: The Event Backbone

Kafka is a **distributed event log**. Every event ever written is stored (until retention expires). Unlike a queue (where reading removes the message), Kafka lets multiple consumers read the same events independently.

### Key Concepts
| Term | Meaning | Analogy |
|------|---------|--------|
| **Topic** | A named channel for events | A mailbox labeled 'orders' |
| **Producer** | Sends events to a topic | The person mailing letters |
| **Consumer** | Reads events from a topic | The person checking the mailbox |
| **Partition**| How a topic is sharded | Multiple mailboxes for 'orders' |
| **Offset**   | Position in the event stream| Page number in a logbook |

### 1.1 Kafka Producer & Consumer Implementation
Let's build a working Producer that emits e-commerce events, and a Consumer that reads them.


In [ ]:
# Cell 2 — Working Kafka Producer & Consumer Simulation
from queue import Queue

class LocalKafkaTopic:
    """Simulates a Kafka topic with partitions"""
    def __init__(self, num_partitions=3):
        self.partitions = [[] for _ in range(num_partitions)]
        
    def produce(self, key, value):
        # Hash the key to consistently route to the same partition
        partition_id = hash(key) % len(self.partitions)
        offset = len(self.partitions[partition_id])
        
        msg = {
            'key': key, 
            'value': value, 
            'partition': partition_id,
            'offset': offset,
            'timestamp': time.time()
        }
        self.partitions[partition_id].append(msg)
        return msg

    def consume_all(self):
        # In real Kafka, consumers track their own offsets
        all_messages = []
        for p in self.partitions:
            all_messages.extend(p)
        return sorted(all_messages, key=lambda x: x['timestamp'])

# 1. Initialize our 'Kafka Cluster'
clickstream_topic = LocalKafkaTopic(num_partitions=3)

# 2. The Producer (Simulating web traffic)
print("[🏭 Producer] Sending events to Kafka...")
events_to_send = [
    {'user_id': 'u123', 'action': 'view', 'item': 'shoe_1', 'price': 99.99},
    {'user_id': 'u456', 'action': 'view', 'item': 'laptop_2', 'price': 899.00},
    {'user_id': 'u123', 'action': 'add_cart', 'item': 'shoe_1', 'price': 99.99}, # u123 returns
    {'user_id': 'u789', 'action': 'purchase', 'item': 'book_9', 'price': 14.50},
    {'user_id': 'u123', 'action': 'purchase', 'item': 'shoe_1', 'price': 99.99}  # u123 buys
]

for ev in events_to_send:
    # We use user_id as the key to guarantee ordering per user
    clickstream_topic.produce(key=ev['user_id'], value=ev)

# 3. The Consumer (Simulating a Real-Time ML Feature Extractor)
print("\n[👀 Consumer] Reading events from Kafka...")
for msg in clickstream_topic.consume_all():
    val = msg['value']
    print(f"  Partition: {msg['partition']} | Offset: {msg['offset']} | User: {val['user_id']} | Action: {val['action']:<9} | Item: {val['item']}")

print("\n💡 Notice how user 'u123' events all landed in the exact same partition. This guarantees strict ordering!")


---
## 2 · Apache Flink: Stateful Stream Processing

Kafka is just the pipe. **Apache Flink** is the engine that processes data inside the pipe.

| Feature | Flink | Spark Streaming |
|---------|-------|----------------|
| True event-at-a-time | Yes | No (micro-batch) |
| Event time handling | Native | Bolt-on |
| Exactly-once | Built-in | Requires config |
| SQL Interface | Flink SQL | Structured Streaming |

### Flink SQL & Windowing Concepts
To build real-time ML features (e.g., "number of transactions in the last 5 minutes"), we use **Windowing**.

1. **Tumbling Window:** Fixed, non-overlapping (e.g., 10:00-10:05, 10:05-10:10).
2. **Sliding Window:** Fixed duration, but overlapping (e.g., 5 min window, sliding every 1 min).
3. **Session Window:** Groups events by activity, separated by a gap of inactivity.


In [ ]:
# Cell 3 — Flink Windowing Implementation
from collections import defaultdict

def tumbling_window_aggregation(events, window_size_minutes=5):
    """
    Simulates Flink's TUMBLE() window over a stream of events.
    """
    # Aggregation state: {window_start_time: {user_id: spend_amount}}
    windows = defaultdict(lambda: defaultdict(float))
    
    for event in events:
        dt = event['time']
        # Calculate which 5-minute bucket this belongs to
        minute_bucket = (dt.minute // window_size_minutes) * window_size_minutes
        window_start = dt.replace(minute=minute_bucket, second=0, microsecond=0)
        
        windows[window_start][event['user_id']] += event['amount']
        
    return windows

# Create simulated fast stream of transactions
base_time = datetime(2026, 4, 1, 10, 0, 0)
stream = []
for i in range(20):
    stream.append({
        'user_id': f"user_{random.randint(1, 3)}",
        'amount': round(random.uniform(5.0, 100.0), 2),
        'time': base_time + timedelta(minutes=random.randint(0, 14))
    })
# Sort to simulate arriving in order
stream.sort(key=lambda x: x['time'])

print("[⚡ Flink Process] Aggregating 5-Min Tumbling Windows...")
aggregated_windows = tumbling_window_aggregation(stream, 5)

for window_time in sorted(aggregated_windows.keys()):
    print(f"\n⏰ Window: {window_time.strftime('%H:%M')} to {(window_time + timedelta(minutes=5)).strftime('%H:%M')}")
    for user, total in aggregated_windows[window_time].items():
        print(f"  ↳ {user}: ${total:.2f} total spend (Real-time Feature)")
        
print("\n💡 In Flink SQL, this entire logic is just: GROUP BY TUMBLE(time, INTERVAL '5' MINUTE)")


---
## 3 · CDC with Debezium: Streaming Database Changes

Change Data Capture (CDC) turns your static database into a streaming event source. 
Tools like **Debezium** listen to the database's Write-Ahead Log (WAL) mapping every `INSERT`, `UPDATE`, and `DELETE` into a Kafka event instantly.

| | Direct DB Query (Polling) | CDC Streaming (Debezium) |
|---|---|---|
| Latency | Query interval (e.g. 5 min) | Milliseconds |
| DB load | High (repeated full scans) | Near-zero (reads log) |
| Captures Deletes | No (unless marked) | Yes |

### 3.1 Handling CDC Payloads
A CDC event contains both the `before` state and the `after` state of the row. Let's see how an AI feature store consumes this.


In [ ]:
# Cell 4 — CDC Debezium Simulation

# 1. Simulated CDC Payloads (What Debezium actually outputs to Kafka)
cdc_stream = [
    # Create user
    {'op': 'c', 'table': 'users', 'before': None, 
     'after': {'id': 99, 'name': 'Alice', 'risk_score': 10}},
     
    # Update user (Score changes 10 -> 85)
    {'op': 'u', 'table': 'users', 
     'before': {'id': 99, 'name': 'Alice', 'risk_score': 10}, 
     'after': {'id': 99, 'name': 'Alice', 'risk_score': 85}},
     
    # Delete user
    {'op': 'd', 'table': 'users', 
     'before': {'id': 99, 'name': 'Alice', 'risk_score': 85}, 
     'after': None}
]

# 2. The Feature Store Consumer
live_feature_store = {}

print("[🗄️ Feature Store] Processing CDC Stream...")
for event in cdc_stream:
    op = event['op']
    table = event['table']
    
    if op == 'c':  # Create/Insert
        row = event['after']
        live_feature_store[row['id']] = row
        print(f"  [INSERT] Added {row['name']} to features.")
        
    elif op == 'u':  # Update
        row = event['after']
        live_feature_store[row['id']] = row
        print(f"  [UPDATE] Changed {row['name']}'s risk_score to {row['risk_score']}. AI models now use new score.")
        
    elif op == 'd':  # Delete
        row_id = event['before']['id']
        del live_feature_store[row_id]
        print(f"  [DELETE] Removed user {row_id} from features (GDPR compliance).")

print(f"\nFinal Feature Store State: {live_feature_store}")


---
## 4 · Live Semantic Layer: Real-Time Vector Embeddings

The final piece of the modern AI streaming stack. 

```
Kafka (text events) --> Flink (generate embeddings) --> Vector DB --> RAG System
```

If a new product is added to the database at 10:00:00, CDC captures it, Flink embeds the text, and it's searchable by your RAG LLM at 10:00:01.


In [ ]:
# Cell 5 — Simulating Real-time Text Embedding for RAG

incoming_docs = [
    'New product launched: AI-powered code review tool',
    'Q4 earnings exceeded expectations by 15%',
    'Security patch CVE-2026-1234 deployed to all servers',
]

print('[🧠 Semantic Layer] Real-Time Embedding Pipeline:')
for doc in incoming_docs:
    # In production: embedding = embedding_model.encode(doc)
    embedding = np.random.randn(5) # Simulating a small vector
    norm = np.linalg.norm(embedding)
    
    print(f'  Event: "{doc[:40]}..."')
    print(f'    -> Model output: [{embedding[0]:.2f}, {embedding[1]:.2f}, ... ]')
    print(f'    -> Pushed to Vector DB (Synchronized < 100ms)')


---
## ✅ Knowledge Check

### Q1: Kafka vs Message Queue
What is the fundamental difference between Kafka and a traditional message queue (like RabbitMQ)?

<details>
<summary>Answer</summary>
In a traditional queue, reading a message **removes** it. In Kafka, reading does NOT remove the message -- it stays in the log until retention expires. Multiple consumers can independently read the same events at their own pace.
</details>

### Q2: Watermarks in Flink
What is the purpose of a `WATERMARK` in stream processing?

<details>
<summary>Answer</summary>
It tells Flink how long to wait for late-arriving events. Network delays happen. A watermark of '5 seconds' means Flink will keep a window open 5 seconds past its closing time to accept straggler events before finalizing the aggregate.
</details>

### Q3: CDC Deletes
Why is polling a database for changes incapable of tracking DELETE operations, while CDC handles them perfectly?

<details>
<summary>Answer</summary>
Polling uses `SELECT * WHERE updated_at > last_check`. If a row is deleted, it is gone from the table, so the `SELECT` query simply won't see it. CDC reads the database's internal transaction log (WAL), which explicitly records a "Delete" event.
</details>


---
## 🔨 Practical Practice

### Exercise 1: Kafka Partition Assignment
You have a Kafka topic with 6 partitions and a consumer group with 4 consumers. How are partitions assigned? What happens if consumer #3 crashes?

### Exercise 2: Sliding Window Implementation
Modify the Tumbling Window code (Cell 3) to implement a **Sliding Window**. The window size should be 10 minutes, but it should slide forward (update) every 2 minutes. 

### Exercise 3: Advanced CDC
Given a Debezium CDC payload for an Order, write Python logic that parses the JSON, calculates the price change (if `op` == 'u'), and drops the event if the price difference is less than $1.00.


In [ ]:
# Cell 6 — Exercise Solutions
print('Ex 1: Partition Assignment')
print('  6 partitions, 4 consumers:')
print('  C1 gets P0, P1 | C2 gets P2, P3 | C3 gets P4 | C4 gets P5')
print('  If C3 crashes, Kafka triggers a rebalance. P4 is reassigned to C1, C2, or C4.')

print('\nEx 3: CDC Delta Calculation')
cdc_event = {'op': 'u', 'before': {'id': 1, 'price': 100.0}, 'after': {'id': 1, 'price': 101.50}}
if cdc_event['op'] == 'u':
    diff = abs(cdc_event['after']['price'] - cdc_event['before']['price'])
    if diff >= 1.00:
        print(f"  Valid price change detected: ${diff:.2f}")
    else:
        print("  Change too small, dropping event.")


---
## 🎯 Summary & Key Takeaways

| Concept | What You Learned |
|---------|------------------|
| **Kafka Core** | Topics, Producers, Consumers, and Partitions for ordering |
| **Flink Windowing** | Grouping infinite streams into finite Tumbling/Sliding chunks |
| **CDC (Debezium)** | Emitting real-time DB changes without polling bottlenecks |
| **Live Semantic** | Generating AI embeddings continuously as data arrives |

**Connections:** Real-time features pipeline into Feature Stores → NB11 | Vector embeddings from streams empower Live RAG → NB22

**Next →** `11_feature_engineering.ipynb` — Feature Engineering & Feature Stores
